In [1]:
import requests
import pandas as pd
import json

In [ ]:
def baixar_todos_os_moves():
    print("Baixando dados da PokeAPI (agora com descrições)...")
    
    # Query atualizada pedindo também o 'flavor_text' (descrição oficial do jogo)
    query = """
    query {
      pokemon_v2_move {
        name
        accuracy
        pp
        power
        pokemon_v2_type {
          name
        }
        pokemon_v2_movedamageclass {
          name
        }
        pokemon_v2_moveflavortexts(where: {language_id: {_eq: 9}}, limit: 1) {
          flavor_text
        }
      }
    }
    """
    
    url = 'https://beta.pokeapi.co/graphql/v1beta'
    response = requests.post(url, json={'query': query})
    
    if response.status_code == 200:
        resposta_json = response.json()
        
        # Prevenção de erros caso a PokeAPI mude a estrutura novamente
        if 'errors' in resposta_json:
            print("Erro na Query do GraphQL:")
            print(json.dumps(resposta_json['errors'], indent=2))
            return
            
        data = resposta_json.get('data', {}).get('pokemon_v2_move', [])
        moves_list = []
        
        for move in data:
            # Extração da descrição, removendo quebras de linha estranhas (\n e \f)
            textos = move.get('pokemon_v2_moveflavortexts', [])
            descricao = textos[0].get('flavor_text', '').replace('\n', ' ').replace('\f', ' ') if textos else 'Sem descrição'
            
            moves_list.append({
                'Nome_Ataque': move.get('name', '').replace('-', ' ').title(),
                'Tipo': move.get('pokemon_v2_type', {}).get('name', '').title() if move.get('pokemon_v2_type') else 'Desconhecido',
                'Categoria': move.get('pokemon_v2_movedamageclass', {}).get('name', '').title() if move.get('pokemon_v2_movedamageclass') else 'Status',
                'Poder': move.get('power') if move.get('power') else 0,
                'Precisão': move.get('accuracy') if move.get('accuracy') else '-',
                'PP': move.get('pp') if move.get('pp') else 0,
                'Descrição': descricao
            })
            
        df = pd.DataFrame(moves_list)
        nome_arquivo = 'todos_ataques_pokemon.csv'
        df.to_csv(nome_arquivo, index=False, encoding='utf-8')
        print(f"Sucesso! Arquivo '{nome_arquivo}' gerado com {len(moves_list)} ataques e suas descrições.")
    else:
        print(f"Falha na conexão. Status: {response.status_code}")



            

def baixar_itens_equipaveis():
    print("Conectando à PokeAPI para buscar itens...")
    
    # Query GraphQL
    query = """
    query {
      pokemon_v2_item {
        name
        cost
        pokemon_v2_itemcategory {
          name
        }
        pokemon_v2_itemeffecttexts(where: {language_id: {_eq: 9}}) {
          short_effect
        }
      }
    }
    """
    
    url = 'https://beta.pokeapi.co/graphql/v1beta'
    response = requests.post(url, json={'query': query})
    
    if response.status_code == 200:
        resposta_json = response.json()
        
        # Verifica se a API retornou um erro na query
        if 'errors' in resposta_json:
            print("Erro na Query do GraphQL:")
            print(json.dumps(resposta_json['errors'], indent=2))
            return
            
        data = resposta_json.get('data', {}).get('pokemon_v2_item', [])
        itens_list = []
        
        # Lista de categorias que os Pokémon costumam equipar
        categorias_equipaveis = [
            'held-items', 'choice', 'effort-training', 'bad-held-items', 
            'training', 'plates', 'species-specific', 'type-enhancement', 
            'mega-stones', 'z-crystals', 'jewels', 'dex-completion'
        ]
        
        for item in data:
            nome = item.get('name', '').replace('-', ' ').title()
            custo = item.get('cost', 0)
            
            # Extrair Categoria
            categoria = item.get('pokemon_v2_itemcategory')
            nome_categoria_raw = categoria.get('name', '') if categoria else ''
            nome_categoria = nome_categoria_raw.replace('-', ' ').title() if nome_categoria_raw else 'Desconhecido'
            
            # Extrair Descrição/Efeito
            efeitos = item.get('pokemon_v2_itemeffecttexts', [])
            efeito = efeitos[0].get('short_effect', '').replace('\n', ' ') if efeitos else 'Sem descrição'
            
            # É equipável se a categoria estiver na nossa lista de permitidos, ou se for uma Berry
            is_berry = 'berries' in nome_categoria_raw.lower() or ('medicine' in nome_categoria_raw.lower() and 'berry' in nome.lower())
            is_holdable = nome_categoria_raw.lower() in categorias_equipaveis
            
            if is_holdable or is_berry:
                itens_list.append({
                    'Nome_Item': nome,
                    'Categoria': nome_categoria,
                    'Efeito': efeito,
                    'Custo_Loja': custo
                })
                
        # Gerar o arquivo CSV
        if len(itens_list) > 0:
            df = pd.DataFrame(itens_list)
            nome_arquivo = 'itens_equipaveis_pokemon.csv'
            df.to_csv(nome_arquivo, index=False, encoding='utf-8')
            print(f"Sucesso! Arquivo '{nome_arquivo}' gerado com {len(itens_list)} itens equipáveis.")
        else:
            print("Nenhum item encontrado.")
            
    else:
        print(f"Falha na conexão. Status code: {response.status_code}")
        print("Detalhes:", response.text)



def baixar_itens_batalha():
    print("Buscando a lista oficial de itens com efeito ATIVO em batalha...")
    
    # Busca apenas os itens que possuem a flag de ter efeito em combate
    url_atributo = "https://pokeapi.co/api/v2/item-attribute/holdable-active/"
    resposta = requests.get(url_atributo)
    
    if resposta.status_code != 200:
        print(f"Erro ao acessar a API. Status: {resposta.status_code}")
        return
        
    dados = resposta.json()
    itens_urls = [item['url'] for item in dados.get('items', [])]
    
    print(f"Foram encontrados {len(itens_urls)} itens (incluindo Berries de combate).")
    print("Baixando os detalhes (isso leva cerca de 15 a 30 segundos)...")
    
    itens_list = []
    
    for i, url in enumerate(itens_urls, 1):
        if i % 20 == 0:
            print(f"Processando item {i} de {len(itens_urls)}...")
            
        try:
            resp_item = requests.get(url)
            if resp_item.status_code == 200:
                item_data = resp_item.json()
                
                nome = item_data.get('name', '').replace('-', ' ').title()
                
                # Deixa a busca pela categoria imune a erros também
                categoria_dict = item_data.get('category')
                categoria = categoria_dict.get('name', '').replace('-', ' ').title() if categoria_dict else 'Desconhecido'
                
                # Procura o efeito competitivo do item (em inglês)
                efeito = "Sem descrição"
                for entry in item_data.get('effect_entries', []):
                    if entry.get('language', {}).get('name') == 'en':
                        efeito = entry.get('short_effect', '').replace('\n', ' ')
                        break
                
                # Fallback: Se não tiver "short_effect", pega a descrição comum (flavor text)
                if efeito == "Sem descrição":
                    for entry in item_data.get('flavor_text_entries', []):
                        if entry.get('language', {}).get('name') == 'en':
                            efeito = entry.get('text', '').replace('\n', ' ')
                            break
                            
                itens_list.append({
                    'Nome_Item': nome,
                    'Categoria': categoria,
                    'Efeito_Batalha': efeito
                })
        except Exception as e:
            print(f"Erro inesperado no item {url}: {e}")
            
    if itens_list:
        df = pd.DataFrame(itens_list)
        nome_arquivo = 'itens_competitivos_pokemon.csv'
        df.to_csv(nome_arquivo, index=False, encoding='utf-8')
        print(f"\nSucesso absoluto! Arquivo '{nome_arquivo}' gerado com {len(itens_list)} itens úteis em batalha.")
    else:
        print("Nenhum item processado.")






def baixar_natures():
    print("\n--- Baixando Natures (Naturezas) ---")
    url = "https://pokeapi.co/api/v2/nature/?limit=50" # Existem cerca de 25 natures
    resposta = requests.get(url)
    
    if resposta.status_code != 200:
        print("Erro ao acessar a API de Natures.")
        return
        
    resultados = resposta.json().get('results', [])
    print(f"Encontradas {len(resultados)} Natures. Processando...")
    
    natures_list = []
    
    for item in resultados:
        resp_nature = requests.get(item['url'])
        if resp_nature.status_code == 200:
            dados = resp_nature.json()
            nome = dados.get('name', '').title()
            
            # Pega os status que sobem e descem (Ex: Jolly aumenta Speed e diminui Sp. Atk)
            aumenta = dados.get('increased_stat')
            diminui = dados.get('decreased_stat')
            
            # Se a nature for Neutra (como Hardy), a API retorna 'None' nesses campos
            stat_aumenta = aumenta.get('name', '').replace('-', ' ').title() if aumenta else "Nenhum (Neutra)"
            stat_diminui = diminui.get('name', '').replace('-', ' ').title() if diminui else "Nenhum (Neutra)"
            
            natures_list.append({
                'Nature': nome,
                'Aumenta (+10%)': stat_aumenta,
                'Diminui (-10%)': stat_diminui
            })
            
    if natures_list:
        df = pd.DataFrame(natures_list)
        nome_arquivo = 'naturezas_pokemon.csv'
        df.to_csv(nome_arquivo, index=False, encoding='utf-8')
        print(f"Sucesso! Arquivo '{nome_arquivo}' gerado com perfeição.")






def baixar_habilidades():
    print("\n--- Baixando Abilities (Habilidades) ---")
    # Existem cerca de 300+ habilidades, limit=500 garante que pegaremos todas de uma vez
    url = "https://pokeapi.co/api/v2/ability/?limit=500"
    resposta = requests.get(url)
    
    if resposta.status_code != 200:
        print("Erro ao acessar a API de Habilidades.")
        return
        
    resultados = resposta.json().get('results', [])
    print(f"Encontradas {len(resultados)} Habilidades. Baixando os detalhes (isso leva cerca de 1 minuto)...")
    
    habilidades_list = []
    
    for i, item in enumerate(resultados, 1):
        if i % 50 == 0:
            print(f"Processando habilidade {i} de {len(resultados)}...")
            
        try:
            resp_hab = requests.get(item['url'])
            if resp_hab.status_code == 200:
                dados = resp_hab.json()
                
                nome = dados.get('name', '').replace('-', ' ').title()
                
                # Procura o efeito competitivo (em inglês)
                efeito = "Sem descrição"
                for entry in dados.get('effect_entries', []):
                    if entry.get('language', {}).get('name') == 'en':
                        efeito = entry.get('short_effect', '').replace('\n', ' ')
                        break
                
                # Fallback: Algumas habilidades mais novas (Gen 9) podem não ter o "short_effect" ainda,
                # então pegamos a descrição comum do jogo (flavor text).
                if efeito == "Sem descrição":
                    for entry in dados.get('flavor_text_entries', []):
                        if entry.get('language', {}).get('name') == 'en':
                            texto = entry.get('flavor_text', entry.get('text', ''))
                            efeito = texto.replace('\n', ' ')
                            break
                            
                habilidades_list.append({
                    'Habilidade': nome,
                    'Efeito': efeito
                })
        except Exception as e:
            print(f"Erro inesperado na habilidade {item['name']}: {e}")
            
    if habilidades_list:
        df = pd.DataFrame(habilidades_list)
        nome_arquivo = 'habilidades_pokemon.csv'
        df.to_csv(nome_arquivo, index=False, encoding='utf-8')
        print(f"Sucesso! Arquivo '{nome_arquivo}' gerado com todas as passivas.")


if __name__ == "__main__":
    baixar_todos_os_moves()
    baixar_itens_equipaveis()
    baixar_itens_batalha()
    baixar_natures()
    baixar_habilidades()
    

Baixando dados da PokeAPI (agora com descrições)...
Sucesso! Arquivo 'todos_ataques_pokemon.csv' gerado com 937 ataques e suas descrições.
Conectando à PokeAPI para buscar itens...
Sucesso! Arquivo 'itens_equipaveis_pokemon.csv' gerado com 280 itens equipáveis.
Buscando a lista oficial de itens com efeito ATIVO em batalha...
Foram encontrados 141 itens (incluindo Berries de combate).
Baixando os detalhes (isso leva cerca de 15 a 30 segundos)...
Processando item 20 de 141...
Processando item 40 de 141...
Processando item 60 de 141...
Processando item 80 de 141...
Processando item 100 de 141...
Processando item 120 de 141...
Processando item 140 de 141...

Sucesso absoluto! Arquivo 'itens_competitivos_pokemon.csv' gerado com 141 itens úteis em batalha.

--- Baixando Natures (Naturezas) ---
Encontradas 25 Natures. Processando...
Sucesso! Arquivo 'naturezas_pokemon.csv' gerado com perfeição.

--- Baixando Abilities (Habilidades) ---
Encontradas 374 Habilidades. Baixando os detalhes (isso l